# Notebook 01 — Build the question set
**Paper:** Carnatic Music Assistant (MTAP). **Protocol:** `evaluation-design.md` (pre-locked, Nov 11, 2026). Do not change the construction rule mid-build; amendments go in the design document, dated.

What this notebook does: clones the AAIDC repository, loads the **committed** FAISS index, draws the seeded passage sample per category, and gives you worksheet cells to author the 96 in-scope questions + 12 out-of-scope probes against those passages. The final cell validates the set and freezes `question_set.jsonl` with its SHA-256 hash.

No API key is needed in this notebook.

In [ ]:
# --- Setup: clone the system repository (public) ---
!git clone -q https://github.com/vinurajd/AAIDC.git
%cd AAIDC
!pip install -q langchain langchain-community langchain-huggingface langchain-groq faiss-cpu sentence-transformers pypdf
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('Repo cloned at commit:')
!git log -1 --format='%H %ci %s'

## The Finding A correction (display only in this notebook)

A pre-evaluation code audit (see `fact-refresh-2026-11-11.md`, Finding A) found that `src/tools.py` picks each tool's category by **positional index** into the metadata map `{"Literature", "Krithis", "Raga"}` — so `raga_index_tool` retrieves from **Krithis** and `krithi_tool` from **Raga** (swapped). Notebook 02 applies and asserts the two-line correction (address categories by name, as `multi_search` already does) before any run. It is shown here so the question set is authored against the *intended* tool domains.

In [ ]:
# --- Load the committed FAISS index and inspect the docstore ---
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
store = FAISS.load_local('src/data/car_research_db', emb, allow_dangerous_deserialization=True)
docs = list(store.docstore._dict.values())
print('Total chunks in committed index:', len(docs))
from collections import Counter
print('Chunks per category:', Counter(d.metadata.get('category') for d in docs))
print('Chunks per source file:')
for k, v in Counter(d.metadata.get('source', d.metadata.get('source_file', '?')) for d in docs).items():
    print(' ', v, k)

In [ ]:
# --- Seeded passage sample (seed FIXED by the design document) ---
import random
SEED = 20261111
rng = random.Random(SEED)

def sample_category(cat, n):
    pool = [(i, d) for i, d in store.docstore._dict.items() if d.metadata.get('category') == cat]
    return rng.sample(pool, min(n, len(pool)))

# 40 candidate passages per category: author 24 questions per stratum from these.
SAMPLES = {cat: sample_category(cat, 40) for cat in ['Literature', 'Raga', 'Krithis']}
for cat, items in SAMPLES.items():
    print(f'--- {cat}: {len(items)} candidate passages (chunk ids below) ---')
    for cid, d in items:
        print(cid, '|', str(d.metadata.get('source', ''))[-60:])

## Authoring worksheet

Run the next cell to print one sampled passage at a time, then fill the `ITEMS` list in the following cell. Rules (from the design document — fixed):

- **S1 theory/literature** (24): passages from `Literature`. **S2 raga/melakarta** (24): from `Raga`. **S3 krithis** (24): from `Krithis`. **S4 cross-category** (24): author from a *pair* of passages from two different categories (the second helper cell prints pairs).
- One question per passage; the passage must fully answer it; gold answer 1–3 sentences; record the passage's chunk id as `evidence_chunk_id`.
- Balance per stratum: ~1/3 factoid, ~1/3 definitional, ~1/3 relational. Set `qtype` accordingly.
- No question answerable from the question alone; no reuse of the app's example prompts.
- The 12 probes (`stratum='PROBE'`) are out-of-scope by construction; `gold_answer` states what a correct abstention says, `evidence_chunk_id` is empty.

In [ ]:
# Print passages for authoring (change the index to page through the sample)
cat, idx = 'Literature', 0   # <- edit me
cid, d = SAMPLES[cat][idx]
print('CHUNK ID:', cid)
print('SOURCE:', d.metadata.get('source'))
print(d.page_content)

In [ ]:
# Helper for S4: print a cross-category passage pair
import itertools
cats = ['Literature', 'Raga', 'Krithis']
a, b = cats[0], cats[1]      # <- edit me
ia, ib = 0, 0                # <- edit me
for cat, i in [(a, ia), (b, ib)]:
    cid, d = SAMPLES[cat][i]
    print('=' * 30, cat, 'CHUNK ID:', cid)
    print(d.page_content[:1500])

In [ ]:
# --- AUTHORING: fill this list (96 items + 12 probes). Two dummy examples are marked DELETE. ---
ITEMS = [
  # {'question': '...', 'gold_answer': '...', 'stratum': 'S1', 'qtype': 'factoid', 'evidence_chunk_id': 'xxxx'},
  # {'question': 'DELETE ME', 'gold_answer': 'DELETE', 'stratum': 'S1', 'qtype': 'factoid', 'evidence_chunk_id': 'dummy'},
]
print('Items authored so far:', len(ITEMS))

In [ ]:
# --- Validate and freeze the question set ---
import json, hashlib
from collections import Counter

assert all(i.get('question') and i.get('gold_answer') and i.get('stratum') for i in ITEMS), 'Every item needs question, gold_answer, stratum'
counts = Counter(i['stratum'] for i in ITEMS)
print('Stratum counts:', dict(counts))
assert counts == {'S1': 24, 'S2': 24, 'S3': 24, 'S4': 24, 'PROBE': 12}, 'Design requires 24 per stratum + 12 probes'
qs = [i['question'].strip().lower() for i in ITEMS]
assert len(set(qs)) == len(qs), 'Duplicate questions present'
for i in ITEMS:
    if i['stratum'] != 'PROBE':
        assert i.get('evidence_chunk_id') in store.docstore._dict, f"Unknown evidence chunk: {i.get('evidence_chunk_id')}"

rows = []
for n, i in enumerate(ITEMS, 1):
    d = store.docstore._dict.get(i.get('evidence_chunk_id', '')) if i['stratum'] != 'PROBE' else None
    rows.append({
        'item_id': f"Q{n:03d}", 'stratum': i['stratum'], 'qtype': i.get('qtype', ''),
        'question': i['question'], 'gold_answer': i['gold_answer'],
        'evidence_chunk_id': i.get('evidence_chunk_id', ''),
        'evidence_source': (d.metadata.get('source', '') if d else ''),
        'evidence_category': (d.metadata.get('category', '') if d else ''),
        'evidence_text': (d.page_content if d else ''),
    })
with open('question_set.jsonl', 'w') as f:
    for r in rows: f.write(json.dumps(r) + '\n')
h = hashlib.sha256(open('question_set.jsonl', 'rb').read()).hexdigest()
print('FROZEN: question_set.jsonl  sha256 =', h)
print('Record this hash in the run log (Notebook 02 prints it again for the record).')
from google.colab import files
files.download('question_set.jsonl')